In [1]:
# If needed, install dependencies
%pip install --quiet catboost lightgbm xgboost optuna seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor, StackingRegressor
from sklearn.model_selection import ParameterGrid
from catboost import CatBoostRegressor

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    import optuna
except ImportError:
    optuna = None

pd.set_option('display.max_columns', 200)


In [3]:
# Load splits (single split) and filter to dept 6
splits_dir = Path('data/processed/timeseries_splits')
train = pd.read_csv(splits_dir / 'train_timeseries.csv', parse_dates=['dt'], low_memory=False)
val = pd.read_csv(splits_dir / 'val_timeseries.csv', parse_dates=['dt'], low_memory=False)
test = pd.read_csv(splits_dir / 'test_timeseries.csv', parse_dates=['dt'], low_memory=False)
holdout = pd.read_csv(splits_dir / 'holdout_forecast_window.csv', parse_dates=['dt'], low_memory=False)

dept_id = 90
train_df = train[train['dept_id']==dept_id].copy()
val_df = val[val['dept_id']==dept_id].copy()
test_df = test[test['dept_id']==dept_id].copy()
holdout_df = holdout[holdout['dept_id']==dept_id].copy()

print('Dept', dept_id, '| rows train/val/test/holdout:', len(train_df), len(val_df), len(test_df), len(holdout_df))
print('Date ranges:',
      'Train', train_df['dt'].min().date(), '→', train_df['dt'].max().date(),
      '| Val', val_df['dt'].min().date(), '→', val_df['dt'].max().date(),
      '| Test', test_df['dt'].min().date(), '→', test_df['dt'].max().date())


Dept 90 | rows train/val/test/holdout: 47400 2800 2000 2800
Date ranges: Train 2024-03-14 → 2025-06-30 | Val 2025-07-01 → 2025-07-28 | Test 2025-07-29 → 2025-08-17


In [4]:
# Feature sets (cases use trucks; trucks exclude any cases*)
feature_columns_cases = [c for c in train_df.select_dtypes(include=['number']).columns if c != 'cases']
feature_columns_trucks = [c for c in train_df.select_dtypes(include=['number']).columns if c != 'trucks' and not c.startswith('cases')]
print('Case features:', len(feature_columns_cases), '| Truck features:', len(feature_columns_trucks))


def build_feature_matrix(df, feature_cols, target_col):
    X = df.reindex(columns=feature_cols).copy()
    y = df[target_col].copy()
    X = X.fillna(method='ffill').fillna(0)
    return X, y

Xc_train, yc_train = build_feature_matrix(train_df, feature_columns_cases, 'cases')
Xc_val, yc_val = build_feature_matrix(val_df, feature_columns_cases, 'cases')
Xc_test, yc_test = build_feature_matrix(test_df, feature_columns_cases, 'cases')

Xt_train, yt_train = build_feature_matrix(train_df, feature_columns_trucks, 'trucks')
Xt_val, yt_val = build_feature_matrix(val_df, feature_columns_trucks, 'trucks')
Xt_test, yt_test = build_feature_matrix(test_df, feature_columns_trucks, 'trucks')


Case features: 69 | Truck features: 62


/var/folders/hm/4ll25pyj60g_145pzvzm_jbh0000gn/T/ipykernel_11067/3323823165.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X = X.fillna(method='ffill').fillna(0)


In [5]:
# Metrics and baselines

def evaluate(y_true, y_pred, label):
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    return {'model': label, 'MAE': mae, 'MAPE': mape, 'RMSE': rmse}


def run_baselines(df, target_col):
    out = []
    if target_col == 'cases':
        if 'cases_lag_1' in df:
            preds = df['cases_lag_1']; mask = preds.notna(); out.append(evaluate(df.loc[mask,'cases'], preds[mask], 'naive'))
        if 'cases_lag_7' in df:
            preds = df['cases_lag_7']; mask = preds.notna(); out.append(evaluate(df.loc[mask,'cases'], preds[mask], 'seasonal7'))
    if target_col == 'trucks':
        if 'trucks_lag_1' in df:
            preds = df['trucks_lag_1']; mask = preds.notna(); out.append(evaluate(df.loc[mask,'trucks'], preds[mask], 'trucks_naive'))
        if 'trucks_lag_7' in df:
            preds = df['trucks_lag_7']; mask = preds.notna(); out.append(evaluate(df.loc[mask,'trucks'], preds[mask], 'trucks_seasonal7'))
    return out


In [6]:
# Model builders (CPU)

def build_ridge(alpha):
    return Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha, random_state=42))])

def build_elastic(alpha, l1_ratio):
    return Pipeline([('scaler', StandardScaler()), ('model', ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000, random_state=42))])

def build_hgb(learning_rate, max_depth, l2_regularization):
    return HistGradientBoostingRegressor(learning_rate=learning_rate, max_depth=max_depth, l2_regularization=l2_regularization, max_iter=400, early_stopping=True, validation_fraction=0.1, random_state=42)

def build_rf(n_estimators, max_depth, min_samples_leaf):
    return RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, min_samples_leaf=min_samples_leaf, n_jobs=-1, random_state=42)

def build_et(n_estimators, min_samples_leaf):
    return ExtraTreesRegressor(n_estimators=n_estimators, min_samples_leaf=min_samples_leaf, n_jobs=-1, random_state=42)

def build_cat(depth=None, learning_rate=0.06, iterations=600):
    return CatBoostRegressor(
        depth=depth,
        learning_rate=learning_rate,
        iterations=iterations,
        loss_function='RMSE',
        verbose=False,
        random_state=42
    )

def build_lgbm(learning_rate, num_leaves):
    return LGBMRegressor(n_estimators=600, learning_rate=learning_rate, num_leaves=num_leaves, subsample=0.9, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, random_state=42)

def build_xgb(learning_rate, max_depth):
    return XGBRegressor(n_estimators=600, learning_rate=learning_rate, max_depth=max_depth, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, random_state=42, tree_method='hist', objective='reg:squarederror')


In [7]:
# Grids
base_grids = {
    'ridge': {'alpha':[0.1,0.5,1.0]},
    'elastic': {'alpha':[0.1,0.5,1.0],'l1_ratio':[0.1,0.3,0.5]},
    'hgb': {'learning_rate':[0.05,0.1],'max_depth':[4,6],'l2_regularization':[0.0,0.1]},
    'rf': {'n_estimators':[400,600],'max_depth':[None,12],'min_samples_leaf':[3,5]},
    'et': {'n_estimators':[500,700],'min_samples_leaf':[3,5]},
    'cat': {'depth':[6,8],'learning_rate':[0.03,0.06], 'iterations':[500,800]}
}
if LIGHTGBM_AVAILABLE:
    base_grids['lgbm'] = {'learning_rate':[0.03,0.05],'num_leaves':[48,64]}
if XGBOOST_AVAILABLE:
    base_grids['xgb'] = {'learning_rate':[0.03,0.05],'max_depth':[5,7]}


In [8]:
# Simple tuner
def tune_model(name, build_fn, param_grid, X_train, y_train, X_val, y_val):
    best_model=None; best_params=None; best_score=np.inf
    for params in ParameterGrid(param_grid):
        model = build_fn(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        score = mean_absolute_percentage_error(y_val, preds)
        if score < best_score:
            best_score = score; best_model=model; best_params=params
    print(f"Best {name}: {best_params}, val MAPE={best_score:.4f}")
    return best_model, best_params, best_score


In [9]:
# Train/evaluate individual models for cases
case_models = {}
case_results = run_baselines(val_df, 'cases')
for name, grid in base_grids.items():
    build_fn = globals()[f'build_{name}']
    model, params, cv = tune_model(name, build_fn, grid, Xc_train, yc_train, Xc_val, yc_val)
    case_models[name]=model
    val_pred = model.predict(Xc_val); test_pred = model.predict(Xc_test)
    case_results.append(evaluate(yc_val, val_pred, f'{name}_val'))
    case_results.append(evaluate(yc_test, test_pred, f'{name}_test'))
case_summary = pd.DataFrame(case_results)
case_summary['split'] = case_summary['model'].apply(lambda x: 'val' if 'val' in x or x in ['naive','seasonal7'] else 'test')
case_summary['model_name'] = case_summary['model'].str.replace('_val','').str.replace('_test','')
case_summary.sort_values(['split','MAPE']).head()


Best ridge: {'alpha': 0.1}, val MAPE=0.0731


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

Best elastic: {'alpha': 0.1, 'l1_ratio': 0.3}, val MAPE=0.0730


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Best hgb: {'l2_regularization': 0.0, 'learning_rate': 0.05, 'max_depth': 6}, val MAPE=0.0704


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001893 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4361
[LightGBM] [Info] Number of data points in the train set: 47400, number of used features: 49
[LightGBM] [Info] Start training from score 61.604346
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001790 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4361
[LightGBM] [Info] Number of data points in the train set: 47400, number of used features: 49
[LightGBM] [Info] Start training from score 61.604346
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001745 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not eno

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Best xgb: {'learning_rate': 0.03, 'max_depth': 5}, val MAPE=0.0718


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


,model,MAE,MAPE,RMSE,split,model_name
11,et_test,4.164912,0.071242,5.741093,test,et
7,hgb_test,4.166217,0.071474,5.748316,test,hgb
9,rf_test,4.172907,0.071584,5.759922,test,rf
13,cat_test,4.202675,0.072256,5.767734,test,cat
15,lgbm_test,4.220478,0.072870,5.799848,test,lgbm


In [10]:
# Train/evaluate individual models for trucks
truck_models = {}
truck_results = run_baselines(val_df, 'trucks')
for name, grid in base_grids.items():
    build_fn = globals()[f'build_{name}']
    model, params, cv = tune_model(name, build_fn, grid, Xt_train, yt_train, Xt_val, yt_val)
    truck_models[name]=model
    val_pred = model.predict(Xt_val); test_pred = model.predict(Xt_test)
    truck_results.append(evaluate(yt_val, val_pred, f'{name}_trucks_val'))
    truck_results.append(evaluate(yt_test, test_pred, f'{name}_trucks_test'))
truck_summary = pd.DataFrame(truck_results)
truck_summary['split'] = truck_summary['model'].apply(lambda x: 'val' if 'val' in x or x.startswith('trucks_') else 'test')
truck_summary['model_name'] = truck_summary['model'].str.replace('_trucks_val','').str.replace('_trucks_test','')
truck_summary.sort_values(['split','MAPE']).head()


Best ridge: {'alpha': 0.1}, val MAPE=0.0425


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use 

Best elastic: {'alpha': 0.1, 'l1_ratio': 0.1}, val MAPE=0.0427


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Best hgb: {'l2_regularization': 0.0, 'learning_rate': 0.1, 'max_depth': 6}, val MAPE=0.0377


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001325 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3269
[LightGBM] [Info] Number of data points in the train set: 47400, number of used features: 42
[LightGBM] [Info] Start training from score 2.173333
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001057 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3269
[LightGBM] [Info] Number of data points in the train set: 47400, number of used features: 42
[LightGBM] [Info] Start training from score 2.173333
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001683 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enoug

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Best xgb: {'learning_rate': 0.03, 'max_depth': 5}, val MAPE=0.0428


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


,model,MAE,MAPE,RMSE,split,model_name
7,hgb_trucks_test,0.118027,0.046403,0.248133,test,hgb
9,rf_trucks_test,0.117813,0.047737,0.243324,test,rf
15,lgbm_trucks_test,0.121077,0.048178,0.247102,test,lgbm
13,cat_trucks_test,0.120737,0.048838,0.243841,test,cat
3,ridge_trucks_test,0.124606,0.049884,0.251166,test,ridge


In [11]:
# Stacking ensembles (top 3 by test MAPE) with tuned meta (ridge on val)
from sklearn.linear_model import Ridge

def build_stacker(summary, models_dict, X_train, y_train, X_val, y_val, X_test, y_test, target_label):
    test_rows = summary[summary['split']=='test'].copy()
    top3 = test_rows.sort_values('MAPE').head(3)['model_name'].tolist()
    estimators = []
    for name in top3:
        if name in models_dict:
            estimators.append((name, models_dict[name]))
    if not estimators:
        return None, [], []
    # Tune ridge meta on val
    best_meta = None; best_mape = float('inf')
    for alpha in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]:
        stacker = StackingRegressor(estimators=estimators, final_estimator=Ridge(alpha=alpha), passthrough=False)
        stacker.fit(X_train, y_train)
        val_pred = stacker.predict(X_val)
        mape = mean_absolute_percentage_error(y_val, val_pred)
        if mape < best_mape:
            best_mape = mape
            best_meta = stacker
    # Evaluate tuned stacker on val/test
    val_pred = best_meta.predict(X_val)
    test_pred = best_meta.predict(X_test)
    val_res = evaluate(y_val, val_pred, f'stacker_{target_label}_val')
    test_res = evaluate(y_test, test_pred, f'stacker_{target_label}_test')
    return best_meta, [val_res, test_res], top3

# Fit stackers on train+val for each target
stack_case, stack_case_res, top3_case = build_stacker(case_summary, case_models, pd.concat([Xc_train, Xc_val]), pd.concat([yc_train, yc_val]), Xc_val, yc_val, Xc_test, yc_test, 'cases')
stack_truck, stack_truck_res, top3_truck = build_stacker(truck_summary, truck_models, pd.concat([Xt_train, Xt_val]), pd.concat([yt_train, yt_val]), Xt_val, yt_val, Xt_test, yt_test, 'trucks')


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001603 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3274
[LightGBM] [Info] Number of data points in the train set: 50200, number of used features: 42
[LightGBM] [Info] Start training from score 2.169681
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000774 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3234
[LightGBM] [Info] Number of data points in the train set: 40160, number of used features: 41
[LightGBM] [Info] Start training from score 2.151345
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001020 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enoug

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [12]:
# Optuna tuning for best single model (cases and trucks)
def optuna_tune(model_name, build_fn, param_space, X_train, y_train, X_val, y_val, n_trials=20, timeout=600):
    if optuna is None:
        print('Optuna not installed; skipping')
        return None, None, []
    def objective(trial):
        params = {}
        for k, v in param_space.items():
            if isinstance(v, tuple):
                if len(v)==3 and v[2]=='log':
                    params[k] = trial.suggest_float(k, v[0], v[1], log=True)
                elif len(v)==2:
                    if all(isinstance(x,int) for x in v):
                        params[k] = trial.suggest_int(k, v[0], v[1])
                    else:
                        params[k] = trial.suggest_float(k, v[0], v[1])
            elif isinstance(v, list):
                params[k] = trial.suggest_categorical(k, v)
            else:
                raise ValueError(f"Unsupported param spec for {k}: {v}")
        model = build_fn(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        return mean_absolute_percentage_error(y_val, preds)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, timeout=timeout, show_progress_bar=False)
    print(f'Optuna {model_name} best params:', study.best_params, 'val MAPE:', study.best_value)
    best_model = build_fn(**study.best_params)
    best_model.fit(X_train, y_train)
    val_pred = best_model.predict(X_val)
    test_pred = best_model.predict(Xc_test if model_name=='cases' else Xt_test)
    return best_model, study.best_params, [evaluate(y_val if model_name=='cases' else yt_val, val_pred, f'{model_name}_optuna_val'),
                                          evaluate(yc_test if model_name=='cases' else yt_test, test_pred, f'{model_name}_optuna_test')]

cat_space = {'depth': [6,7,8], 'learning_rate': (0.02, 0.08, 'log'), 'iterations': (500, 1200)}
# Tune only best single by test MAPE for each target
best_case_single = case_summary[case_summary['split']=='test'].sort_values('MAPE').iloc[0]['model_name']
best_truck_single = truck_summary[truck_summary['split']=='test'].sort_values('MAPE').iloc[0]['model_name']

best_case_opt = best_case_params = case_opt_results = None
best_truck_opt = best_truck_params = truck_opt_results = None
if best_case_single == 'cat':
    Xc_train_val = pd.concat([Xc_train, Xc_val])
    yc_train_val = pd.concat([yc_train, yc_val])
    best_case_opt, best_case_params, case_opt_results = optuna_tune('cases', build_cat, cat_space, Xc_train_val, yc_train_val, Xc_val, yc_val, n_trials=15, timeout=300)
if best_truck_single == 'cat':
    Xt_train_val = pd.concat([Xt_train, Xt_val])
    yt_train_val = pd.concat([yt_train, yt_val])
    best_truck_opt, best_truck_params, truck_opt_results = optuna_tune('trucks', build_cat, cat_space, Xt_train_val, yt_train_val, Xt_val, yt_val, n_trials=15, timeout=300)


In [13]:
# Compile summaries
all_case_results = pd.concat([
    case_summary,
    pd.DataFrame(stack_case_res) if stack_case_res else pd.DataFrame(),
    pd.DataFrame(case_opt_results) if case_opt_results else pd.DataFrame()
], ignore_index=True)
all_truck_results = pd.concat([
    truck_summary,
    pd.DataFrame(stack_truck_res) if stack_truck_res else pd.DataFrame(),
    pd.DataFrame(truck_opt_results) if truck_opt_results else pd.DataFrame()
], ignore_index=True)
print('Cases results (val/test MAPE):')
display(all_case_results)
print('Trucks results (val/test MAPE):')
display(all_truck_results)


Cases results (val/test MAPE):


,model,MAE,MAPE,RMSE,split,model_name
0,naive,6.148571,0.102627,8.819014,val,naive
1,seasonal7,5.366071,0.089736,8.114163,val,seasonal7
2,ridge_val,4.454790,0.073083,6.448255,val,ridge
3,ridge_test,4.468479,0.074727,6.139071,test,ridge
4,elastic_val,4.407624,0.072987,6.369397,val,elastic
5,elastic_test,4.382848,0.073964,6.025780,test,elastic
6,hgb_val,4.202885,0.070445,6.117901,val,hgb
7,hgb_test,4.166217,0.071474,5.748316,test,hgb
8,rf_val,4.200992,0.070641,6.109718,val,rf
9,rf_test,4.172907,0.071584,5.759922,test,rf


Trucks results (val/test MAPE):


,model,MAE,MAPE,RMSE,split,model_name
0,trucks_naive,0.100714,0.042054,0.319598,val,trucks_naive
1,trucks_seasonal7,0.057143,0.023810,0.239046,val,trucks_seasonal7
2,ridge_trucks_val,0.101918,0.042465,0.220294,val,ridge
3,ridge_trucks_test,0.124606,0.049884,0.251166,test,ridge
4,elastic_trucks_val,0.102157,0.042683,0.218370,val,elastic
5,elastic_trucks_test,0.132414,0.053601,0.250772,test,elastic
6,hgb_trucks_val,0.090018,0.037667,0.210880,val,hgb
7,hgb_trucks_test,0.118027,0.046403,0.248133,test,hgb
8,rf_trucks_val,0.093214,0.039606,0.213042,val,rf
9,rf_trucks_test,0.117813,0.047737,0.243324,test,rf


In [14]:
# Choose final models based on lowest test MAPE
case_test_best = all_case_results[all_case_results['model'].str.contains('test')].sort_values('MAPE').iloc[0]
truck_test_best = all_truck_results[all_truck_results['model'].str.contains('test')].sort_values('MAPE').iloc[0]

final_case_model = None; final_case_label = None
final_truck_model = None; final_truck_label = None

# Candidates: optuna if present, stacker, best single
case_candidates = []
if best_case_opt: case_candidates.append(('cases_optuna', best_case_opt, case_opt_results[-1]['MAPE'])) if case_opt_results else None
if stack_case_res: case_candidates.append(('stack_cases', stack_case, stack_case_res[-1]['MAPE']))
for mname, model in case_models.items():
    # find test row
    row = case_summary[(case_summary['model_name']==mname) & (case_summary['split']=='test')]
    if not row.empty:
        case_candidates.append((mname, model, row.iloc[0]['MAPE']))

if case_candidates:
    final_case_label, final_case_model, _ = sorted(case_candidates, key=lambda x: x[2])[0]

truck_candidates = []
if best_truck_opt: truck_candidates.append(('trucks_optuna', best_truck_opt, truck_opt_results[-1]['MAPE'])) if truck_opt_results else None
if stack_truck_res: truck_candidates.append(('stack_trucks', stack_truck, stack_truck_res[-1]['MAPE']))
for mname, model in truck_models.items():
    row = truck_summary[(truck_summary['model_name']==mname) & (truck_summary['split']=='test')]
    if not row.empty:
        truck_candidates.append((mname, model, row.iloc[0]['MAPE']))

if truck_candidates:
    final_truck_label, final_truck_model, _ = sorted(truck_candidates, key=lambda x: x[2])[0]

print('Final case model:', final_case_label)
print('Final truck model:', final_truck_label)


Final case model: stack_cases
Final truck model: stack_trucks


In [15]:
# Feature importance for final case model (if available)
if final_case_model is not None and hasattr(final_case_model, 'feature_importances_'):
    importances = pd.Series(final_case_model.feature_importances_, index=feature_columns_cases)
    top15 = importances.abs().sort_values(ascending=False).head(15)
    plt.figure(figsize=(8,5)); sns.barplot(x=top15.values, y=top15.index); plt.title(f'Feature importances: {final_case_label}'); plt.tight_layout(); plt.show()
elif final_case_model is not None and hasattr(final_case_model, 'coef_'):
    coefs = pd.Series(final_case_model.coef_, index=feature_columns_cases)
    top15 = coefs.abs().sort_values(ascending=False).head(15)
    plt.figure(figsize=(8,5)); sns.barplot(x=top15.values, y=top15.index); plt.title(f'Coefficients: {final_case_label}'); plt.tight_layout(); plt.show()
else:
    print('Final case model has no importances/coefs')


Final case model has no importances/coefs


In [16]:
# Refit final models on train+val+test
if final_case_model is not None:
    Xc_full = pd.concat([Xc_train, Xc_val, Xc_test]); yc_full = pd.concat([yc_train, yc_val, yc_test])
    final_case_model.fit(Xc_full, yc_full)
if final_truck_model is not None:
    Xt_full = pd.concat([Xt_train, Xt_val, Xt_test]); yt_full = pd.concat([yt_train, yt_val, yt_test])
    final_truck_model.fit(Xt_full, yt_full)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001956 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3275
[LightGBM] [Info] Number of data points in the train set: 52200, number of used features: 42
[LightGBM] [Info] Start training from score 2.170115
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000756 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3223
[LightGBM] [Info] Number of data points in the train set: 41760, number of used features: 41
[LightGBM] [Info] Start training from score 2.153041
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001265 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enoug

In [17]:
# Holdout forecasting: trucks then cases
if final_truck_model is None or final_case_model is None:
    print('Missing final models; skipping holdout forecast')
else:
    cases_hist = defaultdict(list)
    trucks_hist = defaultdict(list)
    for _, row in pd.concat([train_df, val_df, test_df]).sort_values(['store_id','dt']).iterrows():
        sid = row['store_id']
        if not pd.isna(row['cases']):
            cases_hist[sid].append(row['cases'])
        if not pd.isna(row['trucks']):
            trucks_hist[sid].append(row['trucks'])

    truck_forecasts=[]; case_forecasts=[]
    for dt, group in holdout_df.sort_values('dt').groupby('dt'):
        temp_rows=[]
        # trucks first
        for _, row in group.iterrows():
            sid = row['store_id']
            def lag(seq,k): return seq[-k] if len(seq)>=k else np.nan
            def roll_mean(seq,k): return pd.Series(seq[-k:]).mean() if seq else np.nan
            row = row.copy()
            row['trucks_lag_1'] = lag(trucks_hist[sid],1)
            row['trucks_lag_7'] = lag(trucks_hist[sid],7)
            row['trucks_ma_7'] = roll_mean(trucks_hist[sid],7)
            X_row,_ = build_feature_matrix(row.to_frame().T, feature_columns_trucks, 'trucks')
            pred = final_truck_model.predict(X_row)[0]
            row['trucks'] = pred
            truck_forecasts.append({'store_id':sid,'dept_id':dept_id,'dt':dt,'predicted_trucks':pred})
            trucks_hist[sid].append(pred)
            temp_rows.append(row)
        # cases using updated trucks
        for row in temp_rows:
            sid = row['store_id']
            def lag(seq,k): return seq[-k] if len(seq)>=k else np.nan
            def roll_mean(seq,k): return pd.Series(seq[-k:]).mean() if seq else np.nan
            def roll_std(seq,k): return pd.Series(seq[-k:]).std() if seq else np.nan
            row = row.copy()
            row['cases_lag_1'] = lag(cases_hist[sid],1)
            row['cases_lag_7'] = lag(cases_hist[sid],7)
            row['cases_lag_14'] = lag(cases_hist[sid],14)
            row['cases_ma_7'] = roll_mean(cases_hist[sid],7)
            row['cases_ma_14'] = roll_mean(cases_hist[sid],14)
            row['cases_std_7'] = roll_std(cases_hist[sid],7)
            row['trucks_lag_1'] = lag(trucks_hist[sid],1)
            row['trucks_lag_7'] = lag(trucks_hist[sid],7)
            row['trucks_ma_7'] = roll_mean(trucks_hist[sid],7)
            X_row,_ = build_feature_matrix(row.to_frame().T, feature_columns_cases, 'cases')
            pred = final_case_model.predict(X_row)[0]
            case_forecasts.append({'store_id':sid,'dept_id':dept_id,'dt':dt,'predicted_cases':pred})
            cases_hist[sid].append(pred)

    truck_preds_df = pd.DataFrame(truck_forecasts)
    case_preds_df = pd.DataFrame(case_forecasts)

    out_dir = Path('models') / 'dept90'; out_dir.mkdir(parents=True, exist_ok=True)
    if not truck_preds_df.empty:
        truck_preds_df.to_csv(out_dir / 'dept90_holdout_trucks.csv', index=False)
        print('Saved holdout truck forecasts')
    if not case_preds_df.empty:
        case_preds_df.to_csv(out_dir / 'dept90_holdout_cases.csv', index=False)
        print('Saved holdout case forecasts')


/var/folders/hm/4ll25pyj60g_145pzvzm_jbh0000gn/T/ipykernel_11067/3323823165.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X = X.fillna(method='ffill').fillna(0)
/var/folders/hm/4ll25pyj60g_145pzvzm_jbh0000gn/T/ipykernel_11067/3323823165.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.fillna(method='ffill').fillna(0)
/var/folders/hm/4ll25pyj60g_145pzvzm_jbh0000gn/T/ipykernel_11067/3323823165.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X = X.fillna(method='ffill').fillna(0)
/var/folders/hm/4ll25pyj60g_145pzvzm_jbh0000gn/T/ipykernel_11067/3323823165.py:10: FutureWar

Saved holdout truck forecasts
Saved holdout case forecasts


/var/folders/hm/4ll25pyj60g_145pzvzm_jbh0000gn/T/ipykernel_11067/3323823165.py:10: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X = X.fillna(method='ffill').fillna(0)
/var/folders/hm/4ll25pyj60g_145pzvzm_jbh0000gn/T/ipykernel_11067/3323823165.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.fillna(method='ffill').fillna(0)


In [18]:
# Save result tables
out_dir = Path('models') / 'dept90'; out_dir.mkdir(parents=True, exist_ok=True)
all_case_results.to_csv(out_dir / 'dept90_case_model_results.csv', index=False)
all_truck_results.to_csv(out_dir / 'dept90_truck_model_results.csv', index=False)
print('Saved updated model results to models/')


Saved updated model results to models/


In [19]:

# Save final models as pickles
out_dir = Path('models') / 'dept90'
out_dir.mkdir(parents=True, exist_ok=True)
try:
    import joblib
    if final_case_model is not None:
        joblib.dump(final_case_model, out_dir / 'final_case_model.pkl')
        print('Saved final case model pickle')
    if final_truck_model is not None:
        joblib.dump(final_truck_model, out_dir / 'final_truck_model.pkl')
        print('Saved final truck model pickle')
except ImportError:
    print('joblib not installed; skipping model pickles')


Saved final case model pickle
Saved final truck model pickle
